In [29]:
import math
import time
import multiprocessing
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

try:
    from pysat.solvers import Glucose3
    from pysat.card import CardEnc, EncType
except:
    import subprocess
    subprocess.check_call(
        ["pip", "install", "python-sat", "--break-system-packages", "-q"]
    )
    from pysat.solvers import Glucose3
    from pysat.card import CardEnc, EncType

try:
    from qiskit import QuantumCircuit
    from pytket.extensions.qiskit import qiskit_to_tk, tk_to_qiskit
    from pytket.passes import DefaultMappingPass
    from pytket.architecture import Architecture
except:
    import subprocess
    subprocess.check_call(
        ["pip", "install", "qiskit", "pytket", "pytket-qiskit", "--break-system-packages", "-q"]
    )
    from qiskit import QuantumCircuit
    from pytket.extensions.qiskit import qiskit_to_tk, tk_to_qiskit
    from pytket.passes import DefaultMappingPass
    from pytket.architecture import Architecture


@dataclass
class QuantumGate:
    gate_id: int
    gate_type: str
    qubits: Tuple[int, int]


@dataclass
class CircuitMapping:
    swap_count: int
    swap_locations: List[Tuple[int, Tuple[int, int]]]
    initial_placement: Dict[int, int]
    mapped_gates: List[Tuple[int, Tuple[int, int]]]
    transition_steps: int
    solve_time: float

In [38]:
def build_original_circuit(num_qubits: int, gates: List[QuantumGate]) -> QuantumCircuit:
    qc = QuantumCircuit(num_qubits)
    for g in gates:
        qc.cx(g.qubits[0], g.qubits[1])
    return qc

def build_circuit_from_mapping(num_qubits: int, mapping: CircuitMapping) -> QuantumCircuit:
    qc = QuantumCircuit(num_qubits)
    timeline = []
    for t, (p1, p2) in mapping.swap_locations:
        timeline.append((t, "SWAP", (p1, p2)))
    for t, (q1, q2) in mapping.mapped_gates:
        timeline.append((t, "CX", (q1, q2)))

    timeline.sort(key=lambda x: x[0])

    current_t = -1
    for t, op_type, qubits in timeline:
        if t != current_t:
            qc.barrier()
            current_t = t

        if op_type == "SWAP":
            qc.swap(qubits[0], qubits[1])
        else:
            qc.cx(qubits[0], qubits[1])
    return qc

def heuristic_routing(gates, num_qubits):
    qc = QuantumCircuit(num_qubits)
    layout = list(range(num_qubits))
    for g in gates:
        q1, q2 = g.qubits
        while abs(layout[q1] - layout[q2]) > 1:
            if layout[q1] < layout[q2]:
                i = layout[q1] + 1 if layout[q1] < layout[q2] else layout[q1] - 1
                # Simple swap logic to bring qubits closer
                target_pos = layout.index(layout[q1] + (1 if layout[q2] > layout[q1] else -1))
                idx1 = layout.index(layout[q1])
                qc.swap(idx1, target_pos)
                layout[idx1], layout[target_pos] = layout[target_pos], layout[idx1]
            qc.barrier()
        qc.cx(layout[q1], layout[q2])
        qc.barrier()
    return qc

In [11]:
import math
import time
import multiprocessing
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional
from qiskit import QuantumCircuit

try:
    from pytket.extensions.qiskit import qiskit_to_tk, tk_to_qiskit
    from pytket.passes import DefaultMappingPass
    from pytket.architecture import Architecture
    from pysat.solvers import Glucose3
    from pysat.card import CardEnc, EncType
except ImportError:
    import subprocess
    subprocess.check_call(["pip", "install", "python-sat", "pytket", "pytket-qiskit", "--break-system-packages", "-q"])
    from pytket.extensions.qiskit import qiskit_to_tk, tk_to_qiskit
    from pytket.passes import DefaultMappingPass
    from pytket.architecture import Architecture
    from pysat.solvers import Glucose3
    from pysat.card import CardEnc, EncType

@dataclass
class QuantumGate:
    gate_id: int
    gate_type: str
    qubits: Tuple[int, int]

@dataclass
class CircuitMapping:
    swap_count: int
    swap_locations: List[Tuple[int, Tuple[int, int]]]
    initial_placement: Dict[int, int]
    mapped_gates: List[Tuple[int, Tuple[int, int]]]
    transition_steps: int
    solve_time: float

def build_original_circuit(num_qubits: int, gates: List[QuantumGate]) -> QuantumCircuit:
    qc = QuantumCircuit(num_qubits)
    for g in gates: qc.cx(g.qubits[0], g.qubits[1])
    return qc

def build_circuit_from_mapping(num_qubits: int, mapping: CircuitMapping) -> QuantumCircuit:
    qc = QuantumCircuit(num_qubits)
    timeline = []
    for t, (p1, p2) in mapping.swap_locations: timeline.append((t, "SWAP", (p1, p2)))
    for t, (q1, q2) in mapping.mapped_gates: timeline.append((t, "CX", (q1, q2)))
    timeline.sort(key=lambda x: x[0])
    current_t = -1
    for t, op_type, qubits in timeline:
        if t != current_t: qc.barrier(); current_t = t
        if op_type == "SWAP": qc.swap(qubits[0], qubits[1])
        else: qc.cx(qubits[0], qubits[1])
    return qc

def heuristic_routing(gates, num_qubits):
    qc = QuantumCircuit(num_qubits)
    layout = list(range(num_qubits))
    for g in gates:
        q1, q2 = g.qubits
        while abs(layout[q1] - layout[q2]) > 1:
            target_pos = layout.index(layout[q1] + (1 if layout[q2] > layout[q1] else -1))
            idx1 = layout.index(layout[q1])
            qc.swap(idx1, target_pos)
            layout[idx1], layout[target_pos] = layout[target_pos], layout[idx1]
            qc.barrier()
        qc.cx(layout[q1], layout[q2]); qc.barrier()
    return qc

def create_harder_circuit() -> Tuple[List[QuantumGate], List[Tuple[int, int]]]:
    gate_tuples = [(0,5), (1,4), (2,3), (0,5), (1,4), (0,3), (2,5), (0,4), (1,5), (0,2), (3,5)]
    gates = [QuantumGate(i, 'cx', pair) for i, pair in enumerate(gate_tuples)]
    deps = [(i, i+1) for i in range(len(gates)-1)]
    return gates, deps

def get_tket_heuristic_circuit(num_qubits, gates, coupling):
    qc = build_original_circuit(num_qubits, gates)
    tk_c = qiskit_to_tk(qc)
    arch = Architecture(coupling)
    DefaultMappingPass(arch).apply(tk_c)
    return tk_to_qiskit(tk_c)

def naive_heuristic(gates, coupling):
    swaps = 0
    coupling_set = set(coupling) | set((b, a) for a, b in coupling)
    for g in gates:
        q1, q2 = g.qubits
        if (q1, q2) not in coupling_set: swaps += abs(q1 - q2) - 1
    return swaps

class SATCircuitMapperCorrect:
    def __init__(self, verbose: bool = True):
        self.verbose = verbose
        self.var_counter = 0
    def _next_var(self) -> int: self.var_counter += 1; return self.var_counter
    def encode_circuit_mapping_base(self, l_q, p_q, edges, gates, deps, max_steps=None):
        V, P, n, K = len(l_q), len(p_q), len(gates), len(edges)
        T = max_steps
        clauses, self.var_counter = [], 0
        pi_vars = {(t, i, j): self._next_var() for t in range(T + 1) for i in range(V) for j in range(P)}
        c_vars = {(t, k): self._next_var() for t in range(1, T + 1) for k in range(K)}
        s_vars = {(g, t): self._next_var() for g in range(n) for t in range(T + 1)}
        for t in range(T + 1):
            for i in range(V):
                clauses.append([pi_vars[(t, i, j)] for j in range(P)])
                for j1 in range(P):
                    for j2 in range(j1 + 1, P): clauses.append([-pi_vars[(t, i, j1)], -pi_vars[(t, i, j2)]])
            for j in range(P):
                for i1 in range(V):
                    for i2 in range(i1 + 1, V): clauses.append([-pi_vars[(t, i1, j)], -pi_vars[(t, i2, j)]])
        for t in range(1, T + 1):
            for j in range(P):
                conn = [c_vars[(t, k)] for k, (p1, p2) in enumerate(edges) if p1 == j or p2 == j]
                for i in range(V): clauses.append([-pi_vars[(t - 1, i, j)], pi_vars[(t, i, j)]] + conn)
            for k, (p1, p2) in enumerate(edges):
                for i in range(V):
                    clauses.append([-c_vars[(t, k)], -pi_vars[(t - 1, i, p1)], pi_vars[(t, i, p2)]])
                    clauses.append([-c_vars[(t, k)], -pi_vars[(t - 1, i, p2)], pi_vars[(t, i, p1)]])
        for g_idx, gate in enumerate(gates):
            clauses.append([s_vars[(g_idx, t)] for t in range(T + 1)])
            q1, q2 = gate.qubits
            for t in range(T + 1):
                v_pairs = []
                for p1, p2 in edges:
                    for qa, qb in [(q1, q2), (q2, q1)]:
                        v = self._next_var()
                        clauses.append([-v, pi_vars[(t, qa, p1)]]); clauses.append([-v, pi_vars[(t, qb, p2)]])
                        v_pairs.append(v)
                clauses.append([-s_vars[(g_idx, t)]] + v_pairs)
        for (gi, gj) in deps:
            for tj in range(T + 1):
                for ti in range(tj, T + 1): clauses.append([-s_vars[(gj, tj)], -s_vars[(gi, ti)]])
        swap_vars = [c_vars[(t, k)] for t in range(1, T + 1) for k in range(K)]
        var_maps = {'pi': pi_vars, 'cv': c_vars, 'sv': s_vars, 'T': T, 'V': V, 'P': P, 'edges': edges, 'gates': gates}
        return clauses, var_maps, swap_vars
    def solve(self, l_q, p_q, edges, gates, deps, initial_bound):
        max_steps = len(gates) + initial_bound * 2
        clauses, var_maps, swap_vars = self.encode_circuit_mapping_base(l_q, p_q, edges, gates, deps, max_steps=max_steps)
        solver = Glucose3(incr=True)
        for c in clauses: solver.add_clause(c)
        curr_best, start_time = None, time.time()
        bound = initial_bound
        max_iterations = 6
        iter_count = 0
        while bound >= 0 and iter_count < max_iterations and (time.time() - start_time < 60):
            iter_count += 1
            act_lit = self._next_var()
            card = CardEnc.atmost(lits=swap_vars, bound=bound, top_id=self.var_counter, encoding=EncType.totalizer)
            for c in card.clauses:
                solver.add_clause([-act_lit] + c)
                for lit in c: self.var_counter = max(self.var_counter, abs(lit))
            if solver.solve(assumptions=[act_lit]):
                res = self._decode_solution(solver.get_model(), var_maps)
                res.solve_time = time.time() - start_time
                curr_best = res
                if self.verbose: print(f"  Found {res.swap_count} SWAPs")
                bound = res.swap_count - 1
            else: break
        return curr_best
    def _decode_solution(self, model, vm):
        ms = set(v for v in model if v > 0)
        init = {i: j for i in range(vm['V']) for j in range(vm['P']) if vm['pi'][(0, i, j)] in ms}
        swaps = [(t, vm['edges'][k]) for (t, k), v in vm['cv'].items() if v in ms]
        mapped = []
        for g in range(len(vm['gates'])):
            for t in range(vm['T'] + 1):
                if vm['sv'][(g, t)] in ms:
                    q1_p = next(j for j in range(vm['P']) if vm['pi'][(t, vm['gates'][g].qubits[0], j)] in ms)
                    q2_p = next(j for j in range(vm['P']) if vm['pi'][(t, vm['gates'][g].qubits[1], j)] in ms)
                    mapped.append((t, (q1_p, q2_p)))
        return CircuitMapping(len(swaps), swaps, init, mapped, vm['T'], 0.0)

def run_upgraded_test():
    num_qubits = 6
    gates, deps = create_harder_circuit()
    coupling = [(i, i + 1) for i in range(num_qubits - 1)]
    print("🔵 (1) ORIGINAL CIRCUIT")
    display(build_original_circuit(num_qubits, gates).draw())
    n_swaps = naive_heuristic(gates, coupling)
    print(f"(2) NAIVE HEURISTIC (Est SWAPs: {n_swaps})")
    display(heuristic_routing(gates, num_qubits).draw())
    print("(3) TKET MAPPED CIRCUIT")
    tket_qc = get_tket_heuristic_circuit(num_qubits, gates, coupling)
    display(tket_qc.draw()); h_swaps = tket_qc.count_ops().get('swap', 0)
    print("(4) RUNNING SAT OPTIMIZATION (MAX 60s)...")
    mapper = SATCircuitMapperCorrect(verbose=True)
    initial_bound = max(h_swaps + 4, 8)
    result = mapper.solve(list(range(num_qubits)), list(range(num_qubits)), coupling, gates, deps, initial_bound)
    if result:
        print(" (4) SAT OPTIMIZED CIRCUIT")
        display(build_circuit_from_mapping(num_qubits, result).draw())
        print(f"\nNaive SWAPs: {n_swaps}\nTKET SWAPs:  {h_swaps}\nSAT SWAPs:   {result.swap_count}")
        imp = h_swaps - result.swap_count
        print(f"\nImprovement over TKET:\n    {imp} swaps\n    percentage = {(imp/max(h_swaps,1)*100):.1f}%")

if __name__ == '__main__':
    run_upgraded_test()

🔵 (1) ORIGINAL CIRCUIT


q_0: ──■──────────────■─────────■─────────■─────────■──
       │              │         │         │         │  
q_1: ──┼────■─────────┼────■────┼─────────┼────■────┼──
       │    │         │    │    │         │    │  ┌─┴─┐
q_2: ──┼────┼────■────┼────┼────┼────■────┼────┼──┤ X ├
       │    │  ┌─┴─┐  │    │  ┌─┴─┐  │    │    │  └───┘
q_3: ──┼────┼──┤ X ├──┼────┼──┤ X ├──┼────┼────┼────■──
       │  ┌─┴─┐└───┘  │  ┌─┴─┐└───┘  │  ┌─┴─┐  │    │  
q_4: ──┼──┤ X ├───────┼──┤ X ├───────┼──┤ X ├──┼────┼──
     ┌─┴─┐└───┘     ┌─┴─┐└───┘     ┌─┴─┐└───┘┌─┴─┐┌─┴─┐
q_5: ┤ X ├──────────┤ X ├──────────┤ X ├─────┤ X ├┤ X ├
     └───┘          └───┘          └───┘     └───┘└───┘

☁☁☁ (2) NAIVE HEURISTIC (Est SWAPs: 24)


░     ░     ░     ░       ░     ░     ░       ░       ░       ░      »
q_0: ─X──░──X──░──X──░──X──░───────░─────░─────░───────░───■───░───────░──────»
      │  ░  │  ░  │  ░  │  ░       ░     ░     ░       ░ ┌─┴─┐ ░       ░      »
q_1: ─X──░──┼──░──┼──░──┼──░───────░──X──░──X──░───────░─┤ X ├─░───────░──────»
         ░  │  ░  │  ░  │  ░       ░  │  ░  │  ░       ░ └───┘ ░       ░      »
q_2: ────░──X──░──┼──░──┼──░───────░──X──░──┼──░───■───░───────░───────░───■──»
         ░     ░  │  ░  │  ░       ░     ░  │  ░ ┌─┴─┐ ░       ░       ░ ┌─┴─┐»
q_3: ────░─────░──X──░──┼──░───────░─────░──X──░─┤ X ├─░───────░───────░─┤ X ├»
         ░     ░     ░  │  ░       ░     ░     ░ └───┘ ░       ░       ░ └───┘»
q_4: ────░─────░─────░──X──░───■───░─────░─────░───────░───────░───■───░──────»
         ░     ░     ░     ░ ┌─┴─┐ ░     ░     ░       ░       ░ ┌─┴─┐ ░      »
q_5: ────░─────░─────░─────░─┤ X ├─░─────░─────░───────░───────░─┤ X ├─░──────»
         ░     ░     ░     ░ └───┘ ░     ░     ░       ░       ░ └───┘ ░      »
«      ░     ░     ░       ░     ░     ░     ░     ░       ░     ░       ░    »
«q_0: ─░──X──░──X──░───────░─────░──X──░─────░─────░───────░──X──░───────░──X─»
«      ░  │  ░  │  ░ ┌───┐ ░     ░  │  ░     ░     ░       ░  │  ░       ░  │ »
«q_1: ─░──┼──░──X──░─┤ X ├─░─────░──┼──░──X──░─────░───────░──X──░───────░──X─»
«      ░  │  ░     ░ └─┬─┘ ░     ░  │  ░  │  ░     ░       ░     ░       ░    »
«q_2: ─░──┼──░─────░───■───░──X──░──X──░──X──░──X──░───────░─────░───■───░────»
«      ░  │  ░     ░       ░  │  ░     ░     ░  │  ░       ░     ░ ┌─┴─┐ ░    »
«q_3: ─░──┼──░─────░───────░──X──░─────░─────░──┼──░───────░─────░─┤ X ├─░────»
«      ░  │  ░     ░       ░     ░     ░     ░  │  ░       ░     ░ └───┘ ░    »
«q_4: ─░──X──░─────░───────░─────░─────░─────░──X──░───■───░─────░───────░────»
«      ░     ░     ░       ░     ░     ░     ░     ░ ┌─┴─┐ ░     ░       ░    »
«q_5: ─░─────░─────░───────░─────░─────░─────░─────░─┤ X ├─░─────░───────░────»
«      ░     ░     ░       ░     ░     ░     ░     ░ └───┘ ░     ░       ░    »
«      ░     ░     ░       ░     ░       ░     ░     ░     ░     ░       ░ 
«q_0: ─░─────░─────░───────░──X──░───────░─────░──X──░─────░─────░───────░─
«      ░     ░     ░       ░  │  ░       ░     ░  │  ░     ░     ░       ░ 
«q_1: ─░──X──░──X──░───────░──┼──░───────░─────░──┼──░─────░──X──░───────░─
«      ░  │  ░  │  ░       ░  │  ░       ░     ░  │  ░     ░  │  ░       ░ 
«q_2: ─░──┼──░──X──░───────░──┼──░───■───░─────░──┼──░──X──░──┼──░───────░─
«      ░  │  ░     ░       ░  │  ░ ┌─┴─┐ ░     ░  │  ░  │  ░  │  ░       ░ 
«q_3: ─░──┼──░─────░───────░──┼──░─┤ X ├─░──X──░──X──░──X──░──X──░───────░─
«      ░  │  ░     ░       ░  │  ░ └───┘ ░  │  ░     ░     ░     ░       ░ 
«q_4: ─░──X──░─────░───■───░──X──░───────░──X──░─────░─────░─────░───■───░─
«      ░     ░     ░ ┌─┴─┐ ░     ░       ░     ░     ░     ░     ░ ┌─┴─┐ ░ 
«q_5: ─░─────░─────░─┤ X ├─░─────░───────░─────░─────░─────░─────░─┤ X ├─░─
«      ░     ░     ░ └───┘ ░     ░       ░     ░     ░     ░     ░ └───┘ ░

☁☁☁ (3) TKET MAPPED CIRCUIT


┌───┐                                                        ┌───┐
node_0: ┤ X ├────────────X───────────────────────────────────────────┤ X ├
        └─┬─┘            │                 ┌───┐     ┌───┐           └─┬─┘
node_1: ──■─────────■────X─────────────────┤ X ├─────┤ X ├──■──────X───■──
        ┌───┐┌───┐┌─┴─┐┌───┐     ┌───┐┌───┐└─┬─┘┌───┐└─┬─┘┌─┴─┐    │      
node_2: ┤ X ├┤ X ├┤ X ├┤ X ├─────┤ X ├┤ X ├──■──┤ X ├──■──┤ X ├─X──X──────
        └─┬─┘└─┬─┘├───┤└─┬─┘┌───┐└─┬─┘└─┬─┘     └─┬─┘     └───┘ │         
node_3: ──■────■──┤ X ├──■──┤ X ├──■────■─────────■────■────────X─────────
                  └─┬─┘     └─┬─┘                    ┌─┴─┐                
node_4: ──■────■────■─────────■────X─────────────────┤ X ├────────────────
        ┌─┴─┐┌─┴─┐                 │                 └───┘                
node_5: ┤ X ├┤ X ├─────────────────X──────────────────────────────────────
        └───┘└───┘

☁☁☁ (4) RUNNING SAT OPTIMIZATION (MAX 60s)...
  Found 8 SWAPs
  Found 7 SWAPs
  Found 6 SWAPs
  Found 5 SWAPs
☁☁☁ (4) SAT OPTIMIZED CIRCUIT


░ ┌───┐ ░       ░       ░ ┌───┐ ░       ░       ░     ░       ░       ░ »
q_0: ─░─┤ X ├─░───────░───────░─┤ X ├─░───────░───────░──X──░───────░───────░─»
      ░ └─┬─┘ ░       ░       ░ └─┬─┘ ░       ░ ┌───┐ ░  │  ░ ┌───┐ ░       ░ »
q_1: ─░───■───░───────░───────░───■───░───X───░─┤ X ├─░──X──░─┤ X ├─░───────░─»
      ░       ░       ░ ┌───┐ ░       ░   │   ░ └─┬─┘ ░     ░ └─┬─┘ ░       ░ »
q_2: ─░───────░───────░─┤ X ├─░───────░───X───░───■───░──X──░───■───░───X───░─»
      ░       ░       ░ └─┬─┘ ░       ░       ░       ░  │  ░       ░   │   ░ »
q_3: ─░───────░───────░───■───░───────░───────░───────░──X──░───X───░───X───░─»
      ░       ░       ░       ░       ░       ░       ░     ░   │   ░       ░ »
q_4: ─░───────░───■───░───────░───────░───■───░───────░─────░───X───░───■───░─»
      ░       ░ ┌─┴─┐ ░       ░       ░ ┌─┴─┐ ░       ░     ░       ░ ┌─┴─┐ ░ »
q_5: ─░───────░─┤ X ├─░───────░───────░─┤ X ├─░───────░─────░───────░─┤ X ├─░─»
      ░       ░ └───┘ ░       ░       ░ └───┘ ░       ░     ░       ░ └───┘ ░ »
«           ░       ░      
«q_0: ──────░───────░───■──
«     ┌───┐ ░       ░ ┌─┴─┐
«q_1: ┤ X ├─░───────░─┤ X ├
«     └─┬─┘ ░       ░ └───┘
«q_2: ──■───░───────░──────
«           ░ ┌───┐ ░      
«q_3: ──────░─┤ X ├─░──────
«           ░ └─┬─┘ ░      
«q_4: ──────░───■───░──────
«           ░       ░      
«q_5: ──────░───────░──────
«           ░       ░


Naive SWAPs: 24
TKET SWAPs:  4
SAT SWAPs:   5

Improvement over TKET:
    -1 swaps
    percentage = -25.0%
